# Market Scanner V2 (Headless Overnight)

Run the cells below to connect this Colab notebook to your GitHub repository and run the overnight scan using Google's cloud compute.

### Features
- **Multi-Factor Scoring**: Trend, Momentum, Options Flow, LEAPS, Fundamentals, Volume
- **Historical Regime Matching**: Finds past trading days with similar technical profiles and computes average forward returns
- **Historical Backtest Filtering**: Calculates 30-day and 90-day win-rates for a bullish setup across each stock's history
- **Neural Network Forecasting**: Custom PyTorch model blended 50/50 with statistical percentile forecasts
- **Causal Discovery, Bayesian Inference, FinBERT Sentiment**: Full ML pipeline
- **Historical Risk Framework**: Drawdown, volatility, daily expected shortfall, beta, liquidity, stress sensitivity, and sizing
- **Fidelity Funds**: Asset-aware analysis for Fidelity mutual funds and Fidelity ETFs/ETPs
- **PHLX AI Semiconductors**: Explicit ASOX constituent coverage and peer tagging
- **Optional Multi-Agent LLM Review**: On-demand critical review in the dashboard (not required for the overnight scan)

Results are saved to `scans_data.db` (SQLite), which you can download or sync to Google Drive for the V2 Streamlit dashboard.

## 1. Setup Environment

In [ ]:
!if [ ! -d /content/market-scanner-pro/.git ]; then git clone https://github.com/dushyant-mishra/market-scanner-pro.git /content/market-scanner-pro; fi
%cd /content/market-scanner-pro
!git pull --ff-only
!pip install -q -r requirements.txt

# Optional LLM setup: add OPENAI_API_KEY in Colab Secrets, then uncomment.
# import os
# from google.colab import userdata
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
# os.environ['OPENAI_REVIEW_MODEL'] = 'gpt-5.6-sol'

# Optional: Mount Google Drive to save the DB permanently
# from google.colab import drive
# drive.mount('/content/drive')
# !ln -s /content/drive/MyDrive/scans_data.db /content/market-scanner-pro/scans_data.db

## 2. Train Neural Network Model

This step trains the custom PyTorch neural network on S&P 500 historical data.
The trained weights are saved to `models/nn_weights.pt` and used by the forecaster
to blend NN predictions with statistical percentile forecasts (50/50 blend).

**You only need to run this once** — or re-run periodically to retrain on fresh data.

In [ ]:
!python run.py train

## 3. Run Overnight Scan

This scans the full S&P 500 + Nasdaq 100 universe, the PHLX US AI Semiconductor (ASOX) basket, and the built-in Fidelity mutual-fund and Fidelity ETF universes with:
- Technical indicators & pattern recognition
- Historical regime matching & backtest win-rate calculation
- Causal discovery, Bayesian inference, FinBERT sentiment
- NN-blended probabilistic price forecasts
- Options strategy ranking where options apply
- Asset-aware fund scoring (NAV history, verified costs, longer-term returns, and downside risk)
- Historical risk analysis and risk-adjusted research ranking

All results are saved to `scans_data.db`. Mutual funds automatically skip inapplicable earnings, corporate-statement, options-flow, and intraday volume-causality steps. Runtime can exceed one hour due to API rate limiting.

In [ ]:
# Start with a genuinely fresh database on repeat notebook runs.
from pathlib import Path
scan_db = Path('scans_data.db')
if scan_db.exists():
    scan_db.unlink()

!python run.py scan

## 4. Download Results

Download both the fresh SQLite scan database and trained neural-network weights. A ZIP bundle is also created for convenient transfer.

In [ ]:
# Validate artifacts before download
from pathlib import Path
import json
import math
import shutil
import sqlite3
db_path = Path('scans_data.db')
weights_path = Path('models/nn_weights.pt')
assert db_path.exists() and db_path.stat().st_size > 0, 'Scan database was not created.'
assert weights_path.exists() and weights_path.stat().st_size > 0, 'NN weights were not created.'

# Fail closed instead of downloading malformed neural forecasts.
with sqlite3.connect(db_path) as connection:
    assert connection.execute('PRAGMA integrity_check').fetchone()[0] == 'ok', 'Database integrity check failed.'
    rows = connection.execute('SELECT ticker, raw_json FROM scan_raw_data').fetchall()
assert rows, 'Scan database contains no detailed results.'
for ticker, raw_json in rows:
    forecasts = json.loads(raw_json).get('forecast', {}).get('forecasts', {})
    for horizon, scenario in forecasts.items():
        values = [scenario.get(name) for name in ('bear', 'base', 'bull')]
        assert all(isinstance(value, (int, float)) and math.isfinite(value) for value in values), f'{ticker} {horizon}d forecast is non-finite: {values}'
        assert values[0] <= values[1] <= values[2], f'{ticker} {horizon}d forecast is unordered: {values}'
print(f'Validated {len(rows)} scan records: database and forecasts are safe to download.')
shutil.make_archive('market_scanner_fresh_artifacts', 'zip', root_dir='.', base_dir='scans_data.db')
# Add weights to the same ZIP without duplicating the database.
import zipfile
with zipfile.ZipFile('market_scanner_fresh_artifacts.zip', 'a', zipfile.ZIP_DEFLATED) as archive:
    archive.write(weights_path, arcname='models/nn_weights.pt')

# Download database, neural-network weights, and combined bundle
from google.colab import files
files.download('scans_data.db')
files.download('models/nn_weights.pt')
files.download('market_scanner_fresh_artifacts.zip')

# Or copy to Google Drive (if mounted)
# !cp scans_data.db /content/drive/MyDrive/scans_data.db
# print('Database saved to Google Drive!')